# Análisis Topológico de la Ingesta de Ácido Fólico en Mujeres Chilenas
## Semanas 1 y 2 — Comprensión de Datos, EDA, Clustering Clásico y TDA Mapper

---

**Equipo 5** | Mariel Álvarez · Viviana · Ana · Álvaro · Jorge  
**Curso:** Uso de Geometría y Topología para la Ciencia de Datos  
**Fecha:** Mayo – Junio 2026  

---

### Referencia al Plan de Trabajo

Este notebook implementa las tareas de las **Semanas 1 y 2** del plan de trabajo del Equipo 5:

| Semana | Tareas cubiertas |
|--------|------------------|
| **Semana 1** | T1.1 Revisión de diccionarios y variables · T1.2 EDA · T1.3 Estrategia de aislamiento de covariables · T1.4 Diseño de pipeline |
| **Semana 2** | T2.1 Preprocesamiento para TDA · T2.2 Clustering clásico · T2.3 Mapper lente UMAP-AF · T2.4 Mapper lente salud neonatal · T2.5 Interpretación · T2.6 Homología persistente |

### Objetivo principal

> Caracterizar subpoblaciones de mujeres embarazadas con distintos patrones de ingesta de ácido fólico (AF) y resultados materno-infantiles, aislando el efecto de variables socioeconómicas y de salud materna.

### Estructura del notebook

```
0. Configuración e importaciones
── SEMANA 1 ──────────────────────────────────────────
1. Carga de datos y revisión de diccionarios    (T1.1)
2. Análisis Exploratorio de Datos (EDA)         (T1.2)
   2.1 Grupos de variables
   2.2 Distribuciones univariadas
   2.3 Análisis de valores faltantes
   2.4 Detección de valores atípicos
   2.5 Correlaciones
   2.6 Análisis bivariado AF – Desenlaces
3. Estrategia de aislamiento de covariables     (T1.3)
4. Diseño y resumen del pipeline TDA            (T1.4)
── SEMANA 2 ──────────────────────────────────────────
5. Preprocesamiento final para TDA              (T2.1)
6. Clustering clásico (K-Means + Ward)          (T2.2)
7. Mapper con lente UMAP sobre ingesta de AF    (T2.3)
8. Mapper con lente de salud neonatal           (T2.4)
9. Interpretación y comparación de Mappers      (T2.5)
10. Homología persistente                        (T2.6)
11. Resumen de hallazgos y próximos pasos
```


## Cargar datos y revisar variables

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('data/dataset2_limpio.csv')


print(f'Dataset: {df.shape[0]:,} registros × {df.shape[1]} variables')
print('Variables y tipos de dato:')
for col in df.columns:
    print(f'- {col}: {df[col].dtype}')

## Agrupar variables en categorias

In [ ]:
VARS_SOCIO_D2 = [ # Datos sociodemográficos de la madre
    'Edad madre', 'Nacionalidad', 'Agrupación nacionalidad',
    'Región', 'Educación', 'N° embarazo'
]

VARS_ANTROPOMETRIA_D2 = [ # Datos antropométricos de la madre
    'KG inicio mamá', 'KG fin mamá', 'Dif peso mamá',
    'Estatura mamá', 'IMC antes', 'IMC después', 'Dif IMC'
]

VARS_GESTACION_D2 = [ # Datos relacionados con la gestación
    'Período', 'Sexo hijo',
    'EG hijo (sem)', 'Condición mamá'
]

VARS_AF_SUPL_D2 = [ # Datos relacionados con el consumo de suplementos y multivitamínicos de AF
    '¿Consume suplementos y/o multivitamínico de AF en embarazo?',
    '¿Consume SAF?',
    "¿Consume SAF+OSAF?",
    'Código SAF',
    'mgAF/cáp SAF+OSAF',
    '¿Cuántos días de SAF?',
    '¿Cuántas cáp SAF?',
    'mgAF/día SAF',
    '¿Consumió MAF?',
    "¿Consumió MAF completo?",
    'Código MAF',
    'mgAF/cáp MAF',
    '¿Cuántos días de MAF?',
    '¿Cuántas cáp MAF?',
    'mgAF/día MAF 1°T',
    'Código OSAF',
    'mgAF/cáp OSAF',
    '¿Cuántos días de OSAF?',
    '¿Cuántas cáp OSAF?',
    "mgAF/día OSAF 1°T",
    'mgAF/día SAF+OSAF 2°T',
    'mgAF/día Suplementos y multivitamínico 1°T',
    'mg/día DFE suplementos y multivitamínico 1°T',
    'Días consumo MAF',
    'mgAF/día período MAF',
    'Días consumo suplementosAF',
    'mgAF/día total período SAF+OSAF',
    'TOTAL AF mg suplementos y multivitamínicos período ',
    'Código consumo SIN'
]

VARS_AF_TIMING_D2 = [ # Datos relacionados con el momento de consumo de suplementos y multivitamínicos de AF
    'Período consumo SAF+OSAF',
    'Consumo OSAF 1° toma',
    'Consumo OSAF 2° toma',
    'Consumo OSAF completo'
]

VARS_DIETA_D2 = [ # Datos relacionados con la dieta de la madre
    'Código alimentos',
    '¿Consumió pan?',
    'Código pan',
    'mgAF/unidad pan',
    'Código pan/día consumido',
    'mg/d AF total pan',
    'mg/d DFE total pan',
    'Total mg/d AF  suple y pan',
    'Total mg/d DFE suple y pan',
    'Consumo suple y pan y pastas (si/no)'
]

VARS_OUTCOME_D2 = [ # Datos relacionados con los resultados de salud del hijo
    'PN hijo (g)',
    '¿Hijo nace c/problema de salud?',
    'Patología RN'
]

In [ ]:
descripcion_columnas = {
    'n' : 'Identificador encuestado (numerico)',
    'Fecha Encuesta' : 'Fecha en que se realizó la encuesta (fecha)',
    "Edad madre" : 'Edad de la madre al momento de la encuesta (numerico)',
    "Nacionalidad" : 'Nacionalidad de la madre (nominal 1-7)',
    'Agrupación nacionalidad' : "nominal (1-2)",
    'Región' : "Región de residencia (nominal 1-16)",
    'Educación' : 'Nivel educacional de la madre (nominal 1-7)',
    'FN hijo' : 'Fecha de nacimiento de ultimo hijo (fecha)',
    'Período' : 'no se sabe nominal (1-2)',
    'N° embarazo' : 'numero del ultimo embarazo (numerico)',
    'Sexo hijo' : 'nominal (1-3)',
    'PN hijo (g)' : 'peso de hijo al nacer en gramos (numerico)',
    'EG hijo (sem)' : 'edad gestacional del hijo en semanas (numerico)',
    'KG inicio mamá' : 'peso en kilogramos de la madre al iniciar el embarazo (numerico)',
    'KG fin mamá' : 'peso en kilogramos de la madre al terminar su embarazo (numerico)',
    'Dif peso mamá' : 'Diferencia de peso de madre en kilogramos (numerico)',
    'Estatura mamá' : 'estatura mama en centimetros (numerico)',
    'IMC antes' : 'Indice de masa corporal de la madre antes del embarazo (numerico)',
    'IMC después' : 'Indice de masa corporal de la madre despues del embarazo (numerico)',
    'Dif IMC' : 'diferencia en IMC madre (numerico)',
    'Condición mamá' : 'condiciones a las que pudo estar expuesta la madre antes o durante el embarazo (nominal 1-3)',
    '¿Hijo nace c/problema de salud?' : 'responde si el hijo nacio con algun problema de salud (binario)',
    'Patología RN' : 'si la respuesta anterior fue si especifica la condición (nominal 0-10 con 0=no)',
    '¿Consume suplementos y/o multivitamínico de AF en embarazo?' : 'responde si se consumio algun tipo de suplemento de acido folico o multivitaminico durante el embarazo (binario)',
    '¿Consume SAF?' : 'responde si se consumen suplementos de acido folico (binario)',
    '¿Consume SAF+OSAF?' : 'responde si se consumen sumplementos de acido folico y otros suplementos de acido folico (binario)',
    'Código SAF' : 'especifica que suplemento se consumio (nominal 0-42 con 0 = no consumio)',
    'mgAF/cáp SAF+OSAF' : 'miligramos de acido folico por capsula de suplementos (numerico)',
    '¿Cuántos días de SAF?' : 'responde cuantos dias a la semana se consumio suplemento de acido folico (numerico 1-7)',
    '¿Cuántas cáp SAF?' : 'cantidad de capsulas de suplemento de acido folico consumido al dia (nominal 1-3)',
    'mgAF/día SAF' : 'miligramos de acido folico al dia en suplementos de acido folico (numerico)',
    '¿Consumió MAF?' : 'responde si consumio Multivitamínicos con AF (binario)',
    '¿Consumió MAF completo?' : 'responde si consumio Multivitamínicos con AF  completo (binario)',
    'Código MAF' : 'especifica multivitaminico con acido folico (nominal 0-42 con 0 = no consumio)',
    'mgAF/cáp MAF' : 'miligramos de acido folico por capsula de multivtamincio (numerico)',
    '¿Cuántos días de MAF?' : 'responde cuantos dias a la semana se consumio multivitaminico con acido folico (numerico 1-7)',
    '¿Cuántas cáp MAF?' : 'cantidad de capsulas de multivitaminico consumidas al dia (nominal 1-3)',
    'mgAF/día MAF 1°T' : 'miligramos al dia de acido folico en multivitaminico durante primer tercio (numerico)',
    'Consumo OSAF 1° toma' : 'consumo de otros suplementos de acido folico durante el primer tercio (binario)',
    'Consumo OSAF 2° toma' : 'consumo de otros suplementos de acido folico durante el segundo tercio (binario)',
    'Consumo OSAF completo' : 'consumo de otros suplementos de acido folico durante el periodo completo (binario)',
    'Código OSAF' : 'especifica tipo de otro suplemento con acido folico (nominal 0-42 con 0 = no consumio)',
    'mgAF/cáp OSAF' : 'miligramos por capsula de folico en capsula de OSAF (numerico)',
    '¿Cuántos días de OSAF?' : 'responde cuantos dias a la semana se consumio OSAF (numerico 1-7)',
    '¿Cuántas cáp OSAF?' : 'cantidad de capsulas de OSAF consumidas al dia (nominal 1-3)',
    'mgAF/día OSAF 1°T' : 'miligramos al dia de acido folico en OSAF durante primer tercio (numerico)',
    'mgAF/día SAF+OSAF 2°T' : 'miligramos al dia de acido folico en OSAF durante segundo tercio (numerico)',
    'mgAF/día Suplementos y multivitamínico 1°T' : 'miligramos al dia de acido folico en suplementos y multivitaminicos durante primer tercio (numerico)',
    'mg/día DFE suplementos y multivitamínico 1°T' : 'no se sabe (numerico)',
    'Período consumo SAF+OSAF' : 'periodo del embarazo en el que se consumieron suplementos de acido folico (nominal 1-3)',
    'Días consumo MAF' : 'total dias que se consumio multivitaminico de acido folico (numerico)',
    'mgAF/día período MAF' : 'miligramos de acido folico en multivitaminico consumidos por dia (numerico)',
    'Días consumo suplementosAF' : 'total dias que se consumio suplemento de acido folico (numerico)',
    'mgAF/día total período SAF+OSAF' : 'miligramos de acido folico en suplemento consumidos por dia (numerico)',
    'TOTAL AF mg suplementos y multivitamínicos período ' : 'total de acido folico consumido en suplementos y multivitaminicos (numerico)',
    'Código consumo SIN' : 'especifica suplementos sin acido folico (nominal 0-42)',
    'Código alimentos' : 'especifica alimentos dejados de consumir durante el embarazo (nominal 1-4)',
    '¿Consumió pan?' : 'consumio pan a base de trigo durante el embarazo (binario)',
    'Código pan' : 'especifica el pan consumido (nominal 0-15)',
    'mgAF/unidad pan' : 'miligramos de acido folico por unidad de pan (numerico)',
    'Código pan/día consumido' : 'cantidad de pan consumida por dia (nominal 0.5, 0-5)',
    'mg/d AF total pan' : 'total de acido folico consumido en pan (numerico)',
    'mg/d DFE total pan' : 'no se sabe (numerico)',
    'Total mg/d AF  suple y pan' : 'total de acido folico consumido en suplementos y pan (numerico)',
    'Total mg/d DFE suple y pan' : 'total de ácido fólico consumido en suplementos y pan (numerico)',
    'Consumo suple y pan y pastas (si/no)' : 'consumo de suplementos, pan y pastas durante el embarazo (binario)'
    }

In [ ]:
variables_nominales = []
variables_binarias = []
variables_numericas = []
variables_fechas = []

for col, desc in descripcion_columnas.items():
    if col in df.columns:
        desc_lower = desc.lower()
        if 'nominal' in desc_lower:
            variables_nominales.append(col)
        elif 'binari' in desc_lower:
            variables_binarias.append(col)
        elif 'fecha' in desc_lower:
            variables_fechas.append(col)
        elif 'numerico' in desc_lower or 'numérico' in desc_lower:
            variables_numericas.append(col)
        else:
            # En caso de que no tenga clasificación clara (como "no se sabe"), lo dejamos numérico por defecto si es float/int
            if pd.api.types.is_numeric_dtype(df[col]):
                variables_numericas.append(col)
                
# 1. Tu código de One-Hot Encoding
df_ohe = pd.get_dummies(df, columns=variables_nominales, dtype=int)

# 2. Función para actualizar las listas dinámicamente
def actualizar_lista_ohe(lista_original, df_ohe, vars_nominales):
    lista_actualizada = []
    for var in lista_original:
        if var in vars_nominales:
            # Si la variable fue transformada, buscamos todas las nuevas columnas en df_ohe
            # get_dummies usa por defecto el formato "NombreColumna_Valor"
            prefijo = f"{var}_"
            nuevas_columnas = [col for col in df_ohe.columns if str(col).startswith(prefijo)]
            lista_actualizada.extend(nuevas_columnas)
        else:
            # Si no fue transformada, la mantenemos tal cual
            # (Verificamos que exista por si acaso hubo algún otro filtro)
            if var in df_ohe.columns:
                lista_actualizada.append(var)
    return lista_actualizada

# 3. Aplicamos la función a todas tus agrupaciones
VARS_SOCIO_D2_OHE = actualizar_lista_ohe(VARS_SOCIO_D2, df_ohe, variables_nominales)
VARS_ANTROPOMETRIA_D2_OHE = actualizar_lista_ohe(VARS_ANTROPOMETRIA_D2, df_ohe, variables_nominales)
VARS_GESTACION_D2_OHE = actualizar_lista_ohe(VARS_GESTACION_D2, df_ohe, variables_nominales)
VARS_AF_SUPL_D2_OHE = actualizar_lista_ohe(VARS_AF_SUPL_D2, df_ohe, variables_nominales)
VARS_AF_TIMING_D2_OHE = actualizar_lista_ohe(VARS_AF_TIMING_D2, df_ohe, variables_nominales)
VARS_DIETA_D2_OHE = actualizar_lista_ohe(VARS_DIETA_D2, df_ohe, variables_nominales)
VARS_OUTCOME_D2_OHE = actualizar_lista_ohe(VARS_OUTCOME_D2, df_ohe, variables_nominales)

# Ejemplo para revisar cómo quedó la primera lista
print("Variables Sociodemográficas actualizadas:")
print(VARS_SOCIO_D2_OHE)

## Analisis exploratorio

### Variables numericas

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display # Útil si usas Jupyter Notebook/Google Colab

# 1. Diccionario con tus agrupaciones originales para iterar fácilmente
agrupaciones = {
    'Sociodemográficas': VARS_SOCIO_D2,
    'Antropometría': VARS_ANTROPOMETRIA_D2,
    'Gestación': VARS_GESTACION_D2,
    'Suplementos AF': VARS_AF_SUPL_D2,
    'Timing AF': VARS_AF_TIMING_D2,
    'Dieta': VARS_DIETA_D2,
    'Outcome': VARS_OUTCOME_D2
}

# (Opcional) Configurar el estilo de los gráficos
sns.set_theme(style="whitegrid")

# 2. Iterar sobre cada agrupación
for nombre_grupo, lista_variables in agrupaciones.items():
    
    # Encontrar cuáles variables de esta agrupación son numéricas
    vars_num_grupo = [var for var in lista_variables if var in variables_numericas]
    
    # Si la agrupación no tiene variables numéricas, pasamos a la siguiente
    if not vars_num_grupo:
        print(f"\n{'='*50}")
        print(f"Grupo: {nombre_grupo} - (No hay variables numéricas)")
        continue
        
    print(f"\n{'='*50}")
    print(f"GRUPO: {nombre_grupo.upper()}")
    print(f"{'='*50}")
    
    # Filtrar el dataframe para quedarnos solo con estas columnas numéricas
    # Nota: Puedes usar df o df_ohe si este último conservó las numéricas
    df_grupo = df[vars_num_grupo]
    
    # A) Estadísticas descriptivas
    print("\n--- Estadísticas Descriptivas ---")
    # Usamos display para que se vea bonito en Jupyter, si usas script normal cambia a print()
    display(df_grupo.describe().round(2)) 


In [ ]:
for nombre_grupo, lista_variables in agrupaciones.items():
    
    # Encontrar cuáles variables de esta agrupación son numéricas
    vars_num_grupo = [var for var in lista_variables if var in variables_numericas]
    
    # Si la agrupación no tiene variables numéricas, pasamos a la siguiente
    if not vars_num_grupo:
        print(f"\n{'='*50}")
        print(f"Grupo: {nombre_grupo} - (No hay variables numéricas)")
        continue
        
    print(f"\n{'='*50}")
    print(f"GRUPO: {nombre_grupo.upper()}")
    print(f"{'='*50}")
    
    # Filtrar el dataframe para quedarnos solo con estas columnas numéricas
    # Nota: Puedes usar df o df_ohe si este último conservó las numéricas
    df_grupo = df[vars_num_grupo]
    
    # B) Gráficos (Boxplot e Histograma) por cada variable numérica
    print("\n--- Visualizaciones ---")
    for var in vars_num_grupo:
        # Crear una figura con 1 fila y 2 columnas
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Ignorar nulos para los gráficos
        data_clean = df[var].dropna()
        
        # 1. Boxplot
        sns.boxplot(x=data_clean, ax=axes[0], color='skyblue')
        axes[0].set_title(f'Boxplot: {var}', fontsize=12, fontweight='bold')
        axes[0].set_xlabel(var)
        
        # 2. Histograma
        sns.histplot(x=data_clean, kde=True, ax=axes[1], color='salmon', bins=20)
        axes[1].set_title(f'Histograma: {var}', fontsize=12, fontweight='bold')
        axes[1].set_xlabel(var)
        axes[1].set_ylabel('Frecuencia')
        
        # Ajustar el diseño y mostrar
        plt.tight_layout()
        plt.show()

### Variables categoricas

In [ ]:
from matplotlib import legend
import numpy as np 
# Graficar la frecuencia para variables binarias y nominales
cat_vars = variables_nominales + variables_binarias
num_cat = len(cat_vars)

for nombre_grupo, lista_variables in agrupaciones.items():
    
    # Encontrar cuáles variables de esta agrupación son categóricas (nominales o binarias)
    vars_cat_grupo = [var for var in lista_variables if var in cat_vars]
    
    # Si la agrupación no tiene variables categóricas, pasamos a la siguiente
    if not vars_cat_grupo:
        print(f"\n{'='*50}")
        print(f"Grupo: {nombre_grupo} - (No hay variables nominales o binarias)")
        continue
        
    print(f"\n{'='*50}")
    print(f"GRUPO: {nombre_grupo.upper()}")
    print(f"{'='*50}")
    
    # Filtrar el dataframe para quedarnos solo con estas columnas categóricas
    df_grupo = df[vars_cat_grupo]

    rows_cat = (len(vars_cat_grupo) + 2) // 3  # 3 columnas por fila
    cols = 3

    fig, axes = plt.subplots(rows_cat, cols, figsize=(20, 4 * rows_cat))
    if rows_cat == 1 and cols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    for i, col in enumerate(vars_cat_grupo):
        sns.countplot(data=df_grupo, x=col, ax=axes[i], palette='viridis', hue=col, legend=False)
        axes[i].set_title(col, fontsize=10)
        axes[i].set_ylabel('')
        axes[i].set_xlabel('')
        axes[i].tick_params(axis='x', rotation=45)

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
        
    plt.tight_layout()
    plt.show()


## Correlación entre agrupaciones

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import itertools
import math
import warnings

# Nuevas importaciones para las métricas no lineales
from scipy.stats import spearmanr
from sklearn.feature_selection import mutual_info_regression
import dcor 

agrupaciones_ohe = {
    'Sociodemográficas': VARS_SOCIO_D2_OHE,
    'Antropometría': VARS_ANTROPOMETRIA_D2_OHE,
    'Gestación': VARS_GESTACION_D2_OHE,
    'Suplementos AF': VARS_AF_SUPL_D2_OHE,
    'Timing AF': VARS_AF_TIMING_D2_OHE,
    'Dieta': VARS_DIETA_D2_OHE,
    'Outcome': VARS_OUTCOME_D2_OHE
}

columnas_validas_df = df_ohe.select_dtypes(include=[np.number, bool]).columns.tolist()

grupos_validos = {}
for nombre, variables in agrupaciones_ohe.items():
    # Nos quedamos con las variables que existan en el dataframe y sean numéricas/booleanas
    vars_validas = [v for v in variables if v in columnas_validas_df]
    if vars_validas:
        grupos_validos[nombre] = vars_validas

# 4. Generar combinaciones de grupos
pares_de_grupos = list(itertools.combinations(grupos_validos.keys(), 2))

# Ignorar warnings matemáticos de scipy/numpy cuando hay divisiones por cero por falta de varianza
with warnings.catch_warnings():
    warnings.filterwarnings('ignore')

    # 1. Definir el umbral de interés
    UMBRAL = 0.4

    pares_interesantes = []

    print("Calculando las 4 métricas de relación y filtrando pares...")
    print("Esto puede tomar un momento dependiendo del tamaño del dataset.\n")

    # 2. Buscar relaciones interesantes
    for grupo_x, grupo_y in pares_de_grupos:
        for var_x in grupos_validos[grupo_x]:
            for var_y in grupos_validos[grupo_y]:
                
                # PREVENCIÓN 1: No comparar la variable consigo misma
                if var_x == var_y:
                    continue
                
                # PREVENCIÓN 2: Extraer de forma segura (por si hay columnas duplicadas en df_ohe)
                s_x = df_ohe[var_x]
                s_y = df_ohe[var_y]
                
                # Si hay columnas duplicadas, s_x o s_y serán DataFrames. Nos quedamos con la 1ra columna.
                if isinstance(s_x, pd.DataFrame): s_x = s_x.iloc[:, 0]
                if isinstance(s_y, pd.DataFrame): s_y = s_y.iloc[:, 0]
                
                # Unir ambas series y botar los NaNs
                data_pair = pd.concat([s_x, s_y], axis=1).dropna()
                
                # Renombrar internamente para garantizar que no haya choque de nombres en esta iteración
                data_pair.columns = ['x', 'y']
                
                # Filtro de tamaño y varianza (si todo es 0 o 1, no hay varianza)
                if len(data_pair) < 5 or data_pair['x'].nunique() <= 1 or data_pair['y'].nunique() <= 1:
                    continue
                    
                x_data = data_pair['x'].values
                y_data = data_pair['y'].values
                
                # --- CÁLCULO DE LAS 4 MÉTRICAS ---
                
                # 1. Pearson (Lineal)
                r = np.corrcoef(x_data, y_data)[0, 1]
                if np.isnan(r): r = 0
                    
                # 2. Spearman (Monótona)
                rho, _ = spearmanr(x_data, y_data)
                if np.isnan(rho): rho = 0
                    
                # 3. Correlación de Distancia
                try:
                    dist_corr = dcor.distance_correlation(x_data, y_data)
                except:
                    dist_corr = 0
                    
                # 4. Información Mutua
                try:
                    mi_raw = mutual_info_regression(x_data.reshape(-1, 1), y_data, random_state=42)[0]
                    mi_norm = np.sqrt(1 - np.exp(-2 * mi_raw))
                except:
                    mi_norm = 0
                
                # --- FILTRO MAESTRO ---
                if any(val >= UMBRAL for val in [abs(r), abs(rho), dist_corr, mi_norm]):
                    pares_interesantes.append({
                        'var_x': var_x,
                        'var_y': var_y,
                        'grupo_x': grupo_x,
                        'grupo_y': grupo_y,
                        'r': r,
                        'rho': rho,
                        'dcor': dist_corr,
                        'mi': mi_norm
                    })
    print(f"¡Listo! Se encontraron {len(pares_interesantes)} pares que superan el umbral de {UMBRAL}.")

    # 3. Graficar SOLAMENTE los pares que pasaron el filtro
    if len(pares_interesantes) > 0:
        columnas = 3
        filas = math.ceil(len(pares_interesantes) / columnas)
        
        # Ajustamos la altura para acomodar la caja de texto más grande
        fig, axes = plt.subplots(filas, columnas, figsize=(16, 5.5 * filas))
        
        # Manejar el caso de que solo haya 1 gráfico (axes no sería un array)
        if filas == 1 and columnas == 1:
            axes = [axes]
        else:
            axes = axes.flatten() 
        
        for i, par in enumerate(pares_interesantes):
            ax = axes[i]
            
            # Jitter para que las binarias no se aplasten
            sns.regplot(
                data=df_ohe, 
                x=par['var_x'], 
                y=par['var_y'], 
                ax=ax,
                fit_reg=False, 
                x_jitter=0.05, 
                y_jitter=0.05,
                scatter_kws={'s': 20, 'alpha': 0.6, 'color': '#4C72B0'}
            )
            
            # Título
            ax.set_title(f"{par['grupo_x']} vs {par['grupo_y']}\n{par['var_x']} / {par['var_y']}", fontsize=10)
            
            # Caja de texto con las 4 métricas
            texto_metricas = (
                f"Pearson: {par['r']:.2f}\n"
                f"Spearman: {par['rho']:.2f}\n"
                f"Dist. Corr: {par['dcor']:.2f}\n"
                f"Mut. Info: {par['mi']:.2f}"
            )
            
            ax.annotate(
                texto_metricas, 
                xy=(0.05, 0.95), xycoords=ax.transAxes, 
                ha='left', va='top', fontsize=9, fontweight='bold', 
                bbox=dict(boxstyle="round,pad=0.4", fc="#f8f9fa", ec="gray", alpha=0.9)
            )

        # Limpiar los subplots sobrantes
        for j in range(i + 1, len(axes)):
            fig.delaxes(axes[j])
            
        plt.tight_layout()
        plt.show()
    else:
        print("No se generaron gráficos. Intenta bajar el umbral (ej. a 0.2 o 0.3).")